# Qwen2.5-0.5B: chọn cấu hình quantize để triển khai

Một tình huống thật: bạn cần chạy Qwen2.5-0.5B-Instruct cho một tính năng tiếng Việt trên hai loại máy, laptop GPU 4 GB và một
server **chỉ có CPU**. Có rất nhiều lựa chọn: bf16/fp16, bitsandbytes 8 bit và 4 bit (NF4, FP4, double quant), GPTQ/AWQ tự
cài, GGUF của llama.cpp/Ollama ở nhiều mức. Bạn đo **cùng một thước đo chất lượng** (bpc trên validation Wikipedia tiếng Việt
của module 04/05) cho mọi lựa chọn, đo bộ nhớ và tốc độ trên từng loại máy, rồi viết một bản khuyến nghị có số.

**Cách làm bài**
- Bốn hàm được test chấm (cell dưới). Phần còn lại tự thiết kế.
- bitsandbytes chỉ chạy trên GPU: dùng kernel `ai-gpu` (`setup/gpu`). Không có GPU: bỏ các dòng bitsandbytes, làm phần GGUF
  và CPU, ghi rõ.
- Model: `Qwen/Qwen2.5-0.5B-Instruct` từ Hugging Face (khoảng 1 GB, module 09 đã tải); GGUF lấy từ Ollama
  (`ollama pull qwen2.5:0.5b-instruct-q4_K_M`, `...-q8_0`, `...-fp16`, `...-q3_K_M`); đường dẫn tệp blob đọc từ
  `ollama show --modelfile <tag>` (dòng `FROM`).
- Chấm: `uv run pytest tracks/ai-engineer/modules/10-quantization/exercises/advanced/02-qwen-quantization-deploy -q`

In [ ]:
# explore: giới hạn luồng, tìm gốc repo, kiểm Wikipedia tiếng Việt (tải một lần bằng env gốc), chỗ để checkpoint
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
WIKI = ROOT / "data/raw/wikipedia-vi"
if not (WIKI.is_dir() and any(WIKI.iterdir())):
    from data.download_dataset import ensure

    ensure("wikipedia-vi")
SHARD = WIKI / "20231101.vi/train-00002-of-00004.parquet"
CKPT = Path(os.environ.get("GPT_MINI_CKPT", ROOT / "data/raw/gpt-mini-vi/gpt_mini_vi.pt"))
FALLBACK = Path(os.environ.get("GPT_MINI_FALLBACK", ROOT / "data/raw/gpt-mini-vi/fallback_small.pt"))
print("checkpoint:", CKPT, "có" if CKPT.exists() else "chưa có")

In [ ]:
import gguf
import gguf.quants
import numpy as np
import torch

## 1. Bốn hàm được chấm

**Đo GGUF bằng cùng thước đo.** Ollama không trả log-probability của prompt, nên không tính được perplexity của một GGUF qua
Ollama. Cách làm ở đây: giải lượng tử từng tensor GGUF bằng `gguf.quants.dequantize` (đúng phép tính llama.cpp dùng), nạp vào
model Transformers, rồi đo bpc như mọi dòng khác. Kết quả là chất lượng của **trọng số** GGUF; kernel llama.cpp còn lượng tử
hoá activation khi nhân ma trận trên CPU, nên số đo thật qua llama.cpp có thể lệch thêm một chút (kiểm bằng `llama-cpp-python`
trên một phần validation nếu bạn đã cài).

- `gguf_to_hf_name(name)`: tên tensor GGUF kiến trúc `qwen2` → tên trong `state_dict` của `Qwen2ForCausalLM`:
  `token_embd.weight` → `model.embed_tokens.weight`, `output_norm.weight` → `model.norm.weight`, `output.weight` →
  `lm_head.weight`, và `blk.{i}.{x}.{weight|bias}` → `model.layers.{i}.{y}.{weight|bias}` với `attn_q/k/v` →
  `self_attn.q_proj/k_proj/v_proj`, `attn_output` → `self_attn.o_proj`, `ffn_gate/up/down` → `mlp.gate_proj/up_proj/down_proj`,
  `attn_norm` → `input_layernorm`, `ffn_norm` → `post_attention_layernorm`. Tên khác: `KeyError`.
- `load_gguf_state_dict(path)`: đọc tệp bằng `gguf.GGUFReader`, giải lượng tử mọi tensor, trả `(state, types)`: `state` là tên
  HF → tensor `float32` đúng shape PyTorch (GGUF ghi shape theo thứ tự ngược: `ne0` là chiều trong cùng); `types` là tên HF →
  tên kiểu GGUF (`"Q4_K"`, `"Q8_0"`, `"F32"`...).
- `pareto_front(rows, minimize, maximize=())`: `rows` là list dict có khoá `"name"` và các cột số; trả tên các dòng **không bị
  trội** (dòng khác không tệ hơn ở mọi cột và tốt hơn hẳn ở ít nhất một cột), giữ thứ tự của `rows`.
- `pick_config(rows, max_memory_mib, max_bpc_increase, baseline_bpc, speed_key="tok_s")`: trong các dòng có `"memory_mib"` không
  vượt ngân sách và `"bpc"` không tăng quá `max_bpc_increase` so với `baseline_bpc`, trả tên dòng có `speed_key` lớn nhất (hoà
  thì dòng ít bộ nhớ hơn); không dòng nào đạt thì `None`.

In [ ]:
def gguf_to_hf_name(name: str) -> str:
    # TODO
    raise NotImplementedError


def load_gguf_state_dict(path) -> tuple[dict[str, torch.Tensor], dict[str, str]]:
    # TODO
    raise NotImplementedError


def pareto_front(rows: list[dict], minimize: tuple[str, ...], maximize: tuple[str, ...] = ()) -> list[str]:
    # TODO
    raise NotImplementedError


def pick_config(rows: list[dict], max_memory_mib: float, max_bpc_increase: float, baseline_bpc: float,
                speed_key: str = "tok_s"):
    # TODO
    raise NotImplementedError

## 2. Gợi ý bố cục (không chấm)

1. **Thước đo**: bpc của mỗi cấu hình trên validation 100 bài (cửa sổ trượt như module 05, context 1 024, stride 512). bpc so
   được giữa Qwen và GPT mini dù khác tokenizer.
2. **GPU** (ghi rõ GPU có đang chạy việc khác không): bf16, fp16, bitsandbytes int8, NF4, FP4, NF4 + double quant, RTN/GPTQ int4
   tự cài (fake quant, chỉ đo chất lượng), GGUF giải lượng tử. Mỗi dòng: bpc, bộ nhớ sau khi nạp và đỉnh khi sinh
   (`torch.cuda.max_memory_allocated`), token/giây khi sinh 128 token batch 1.
3. **CPU** (ghi số luồng, máy có bận không): token/giây của GGUF bằng `llama-cpp-python` hoặc Ollama (`num_gpu: 0`), so với
   Transformers float32/bf16 trên CPU.
4. **Đọc kiểu tensor** trong tệp `Q4_K_M` bằng `load_gguf_state_dict`: tensor nào không phải `Q4_K`, vì sao (gợi ý: hidden size
   của Qwen2.5-0.5B và kích thước khối của k-quant)?
5. **Quyết định**: `pareto_front` trên (bpc, bộ nhớ, tốc độ) cho từng máy, `pick_config` cho hai ràng buộc thật, rồi viết
   `REPORT.md`.

In [ ]:
# explore: TODO: pipeline của bạn